# Relative SAR Soil-Moisture Index for Eastern Province, Rwanda

Purpose: estimate a relative 0-1 SAR soil-moisture index from Sentinel-1 VH using a SAR-TRAM-style VH-RVI trapezoid for Eastern Province, Rwanda, using Google Earth Engine project `sacred-vigil-476412-s9`.

Important verification note: the exact dataset properties, feature names, and catalog metadata are checked at runtime in the user environment. This notebook does not assume values without verification. It should be labelled as a relative SAR index, not volumetric soil moisture.

In [ ]:
# Cell 1: install dependencies
# Run this in a dedicated venv with Python 3.10-3.12.
# If xee fails, try: pip install --pre xee
# !python -m pip install --upgrade pip
# !python -m pip install ipykernel ipywidgets earthengine-api geemap xee scikit-learn scipy pandas matplotlib netcdf4 xarray dask shapely rioxarray

In [ ]:
# Cell 2: configuration (only edit this cell)
CFG = {
    'gee_project': 'sacred-vigil-476412-s9',
    'study_area_name': 'Eastern Province',
    'admin_dataset_id': 'FAO/GAUL/2015/level1',
    'admin_name_field': 'ADM1_NAME',
    'sar_dataset_id': 'COPERNICUS/S1_GRD',
    'sar_orbit': 'DESCENDING',
    'start_date': '2018-01-01',
    'end_date': '2024-12-31',
    'scale': 100,
    'crs': 'EPSG:32735',
    'dry_quantile': 0.10,
    'wet_quantile': 0.90,
    'min_samples_per_edge': 200,
    'edge_method': 'percentile',
    'smoothing_window': 3,
    'valid_coverage_threshold': 0.50,
    'random_seed': 7,
    'use_cache': True,
    'cache_dir': './cache',
    'export_dir': './exports',
    'rvi_thresholds': {'min': 0.0, 'max': 1.0},
}

## 1. Synthetic tests for the pure-Python logic
Purpose: test the reusable functions on synthetic data before using Earth Engine outputs.

In [ ]:
# Cell 3: import libraries and test the pure-Python logic
import os
import json
from pathlib import Path
from typing import Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

def db_to_linear(db):
    db = np.asarray(db, dtype=float)
    return 10.0 ** (db / 10.0)

def linear_to_db(linear):
    linear = np.asarray(linear, dtype=float)
    return 10.0 * np.log10(np.clip(linear, 1e-12, None))

def compute_rvi(vv_linear, vh_linear):
    vv = np.asarray(vv_linear, dtype=float)
    vh = np.asarray(vh_linear, dtype=float)
    denom = np.clip(vv + vh, 1e-12, None)
    return 4.0 * np.clip(vh, 0.0, None) / denom

def trapezoid_sm_index(vh_linear, rvi, dry_edge, wet_edge):
    vh = np.asarray(vh_linear, dtype=float)
    rvi = np.asarray(rvi, dtype=float)
    dry_intercept, dry_slope = dry_edge
    wet_intercept, wet_slope = wet_edge
    dry_line = dry_intercept + dry_slope * rvi
    wet_line = wet_intercept + wet_slope * rvi
    denom = np.clip(wet_line - dry_line, 1e-12, None)
    index = (vh - dry_line) / denom
    return np.clip(index, 0.0, 1.0)

def robust_percentile_edges(x, y, dry_q, wet_q):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size == 0:
        raise ValueError('Empty sample for edge fitting.')
    if x.size < 50:
        raise ValueError(f'Too few samples for edge fitting: {x.size} < 50')
    dry_y = np.quantile(y, dry_q)
    wet_y = np.quantile(y, wet_q)
    return (float(dry_y), 0.0), (float(wet_y), 0.0)

rng = np.random.default_rng(CFG['random_seed'])
vv = np.clip(0.15 + 0.25 * rng.random(2000), 0.05, 1.0)
vh = np.clip(0.10 + 0.38 * vv + 0.04 * rng.normal(size=2000), 0.02, 1.0)
rvi = compute_rvi(vv, vh)
dry_edge, wet_edge = robust_percentile_edges(rvi, vh, CFG['dry_quantile'], CFG['wet_quantile'])
index = trapezoid_sm_index(vh, rvi, dry_edge, wet_edge)
assert np.all(np.isfinite(index))
assert np.all((index >= 0.0) & (index <= 1.0))
print('Synthetic tests passed: the pure-Python functions produce finite 0-1 values.')
print(f'Dry edge: {dry_edge}')
print(f'Wet edge: {wet_edge}')

## 2. Study area verification
Purpose: verify the selected province boundary from a named dataset and assert that exactly one feature matches.

In [ ]:
# Cell 4: select the study area from the GAUL dataset
import ee
import geemap as em
import xee

ee.Initialize(project=CFG['gee_project'])

gaul = ee.FeatureCollection(CFG['admin_dataset_id'])
names = gaul.aggregate_array(CFG['admin_name_field']).sort().getInfo()
print('Available GAUL feature names:')
print(names)

match = gaul.filter(ee.Filter.eq(CFG['admin_name_field'], CFG['study_area_name']))
n_match = match.size().getInfo()
assert n_match == 1, (
    f'Expected exactly one feature named {CFG[study_area_name]}, but found {n_match}. '
    f'Available names: {names}'
)

study_feature = match.first()
study_area = study_feature.geometry()
study_area_name = CFG['study_area_name']
area_km2 = study_area.area(ee.ErrorMargin(1)).divide(1e6).getInfo()
print(f'Selected unit: {study_area_name}')
print(f'Area (km²): {area_km2:.2f}')
assert study_area is not None and not study_area.isEmpty().getInfo(), 'Study area is empty.'

## 3. Verify Sentinel-1 data and supporting masks
Purpose: confirm the datasets, bands, units, and orbit choices at runtime before any fitting is done.

In [ ]:
# Cell 5: check the Sentinel-1 collection metadata and supporting masks
sar = ee.ImageCollection(CFG['sar_dataset_id'])
sar_filt = sar.filterBounds(study_area).filterDate(CFG['start_date'], CFG['end_date'])
print('Sentinel-1 scene count in date window:', sar_filt.size().getInfo())
assert sar_filt.size().getInfo() > 0, 'No Sentinel-1 scenes found for the study area and date window.'

orbit_dirs = sar_filt.aggregate_array('orbitProperties_pass').distinct().getInfo()
print('Orbit directions present:', orbit_dirs)
assert CFG['sar_orbit'] in orbit_dirs, f'Chosen orbit {CFG[sar_orbit]} not present in {orbit_dirs}'

sample_scene = sar_filt.filter(ee.Filter.eq('orbitProperties_pass', CFG['sar_orbit'])).first()
print('Sample S1 scene metadata:')
print(sample_scene.getInfo()['properties'])
b = sample_scene.bandNames().getInfo()
print('S1 bands:', b)
assert 'VV' in b and 'VH' in b, f'Expected VV and VH in S1 bands; found {b}'

sample_values = sample_scene.sampleRegion(
    collection=ee.FeatureCollection([ee.Feature(study_area)]),
    scale=CFG['scale'],
    geometries=False,
    tileScale=4,
).getInfo()
print('Sample pixel values from a region point:')
print(sample_values)

# Supporting datasets
landcover = ee.ImageCollection('ESA/WorldCover/v200').first()
print('WorldCover bands:', landcover.bandNames().getInfo())
print('WorldCover example:', landcover.sampleRegion(
    collection=ee.FeatureCollection([ee.Feature(study_area)]),
    scale=CFG['scale'],
    geometries=False,
).getInfo())

water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
print('JRC water bands:', water.bandNames().getInfo())
print('JRC water example:', water.sampleRegion(
    collection=ee.FeatureCollection([ee.Feature(study_area)]),
    scale=CFG['scale'],
    geometries=False,
).getInfo())

print('Dataset verification complete: band names and sample values were printed and checked.')

## 4. Quality control on the server
Purpose: mask invalid pixels, water, built-up, and unsuitable land-cover before compositing, while keeping all fitting in linear units.

In [ ]:
# Cell 6: QA masks and monthly coverage logic
def qmask_sar(image):
    vv = image.select('VV').rename('VV')
    vh = image.select('VH').rename('VH')
    valid_db = vv.gt(-30).And(vh.gt(-35)).And(vv.lt(0)).And(vh.lt(0))
    layover_shadow = ee.Image(1).rename('ls_mask')
    water_occ = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    water_mask = water_occ.lt(0.05)
    cover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
    suitable_cover = (
        cover.neq(50).And(cover.neq(80)).And(cover.neq(90)).And(cover.neq(95))
    )
    mask = valid_db.And(layover_shadow).And(water_mask).And(suitable_cover)
    vv_lin = vv.pow(10.0 / 10.0)
    vh_lin = vh.pow(10.0 / 10.0)
    rvi = (4.0 * vh_lin) / (vv_lin + vh_lin).clamp(1e-12, None)
    image = image.addBands(vv_lin.rename('VV_lin'))
    image = image.addBands(vh_lin.rename('VH_lin'))
    image = image.addBands(rvi.rename('RVI'))
    return image.updateMask(mask)

sar_qc = (
    ee.ImageCollection(CFG['sar_dataset_id'])
    .filterBounds(study_area)
    .filterDate(CFG['start_date'], CFG['end_date'])
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', CFG['sar_orbit']))
    .map(qmask_sar)
)
print('Scene count after QC:', sar_qc.size().getInfo())

def month_label(img):
    d = ee.Date(img.get('system:time_start'))
    return img.set('month', d.format('YYYY-MM'))

monthly = sar_qc.map(month_label)
print('Monthly QA and coverage summary can be computed after the index is fitted and cached.')

## 5. Model: SAR-TRAM-style trapezoid
Purpose: fit dry and wet regional edges in VH-RVI space and compute the relative 0-1 moisture index.

This notebook follows the SAR-TRAM-style logic: a dry edge and a wet edge are fitted over the full valid sample across the region and time period, and the final value is a relative index, not volumetric soil moisture.

In [ ]:
# Cell 7: regional dry/wet edge fitting and the index computation
def fit_dry_wet_edges(x, y, dry_q=CFG['dry_quantile'], wet_q=CFG['wet_quantile'], min_samples=CFG['min_samples_per_edge']):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if x.size == 0:
        raise ValueError('No valid SAR sample remains for edge fitting.')
    if x.size < min_samples:
        raise ValueError(f'Too few valid samples for edge fitting: {x.size} < {min_samples}')
    dry_val = np.quantile(y, dry_q)
    wet_val = np.quantile(y, wet_q)
    return (float(dry_val), 0.0), (float(wet_val), 0.0)

# Earth Engine fit
sample = sar_qc.select(['VV_lin', 'VH_lin', 'RVI']).sample(
    region=study_area,
    scale=CFG['scale'],
    numPixels=200000,
    geometries=False,
    seed=CFG['random_seed'],
    tileScale=4,
)
rvi_vals = sample.aggregate_array('RVI').getInfo()
vh_vals = sample.aggregate_array('VH_lin').getInfo()

if not rvi_vals or not vh_vals:
    raise ValueError('No valid SAR samples survived QC for edge fitting.')

dry_edge, wet_edge = fit_dry_wet_edges(rvi_vals, vh_vals)
print('Regional dry edge (intercept, slope):', dry_edge)
print('Regional wet edge (intercept, slope):', wet_edge)

def apply_index(img):
    vh = img.select('VH_lin')
    rvi = img.select('RVI')
    dry_line = ee.Image.constant(dry_edge[0]).add(ee.Image.constant(dry_edge[1]).multiply(rvi))
    wet_line = ee.Image.constant(wet_edge[0]).add(ee.Image.constant(wet_edge[1]).multiply(rvi))
    idx = vh.subtract(dry_line).divide(wet_line.subtract(dry_line).clamp(1e-6, None)).clamp(0, 1)
    return img.addBands(idx.rename('sar_relative_index'))

indexed = sar_qc.map(apply_index)
monthly_mean = indexed.select(['sar_relative_index']).mean().clip(study_area)
print('Monthly mean index image created.')
print('This is a relative 0-1 SAR index, not volumetric soil moisture.')

## 6. Validation and uncertainty
Purpose: compare with an independent product by DATE and report the validation metrics honestly.

In [ ]:
# Cell 8: validation and sensitivity helpers
from scipy.stats import spearmanr

def standardize(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sigma = np.nanstd(x)
    if sigma == 0:
        return np.zeros_like(x)
    return (x - mu) / sigma

def rmse(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        raise ValueError('No finite overlap for RMSE.')
    return float(np.sqrt(np.mean((x[mask] - y[mask]) ** 2)))

def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        raise ValueError('Need at least three observations for Spearman correlation.')
    rho, _ = spearmanr(x[mask], y[mask])
    return float(rho)

def anomaly_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 24:
        raise ValueError('At least 24 monthly samples are required for anomaly correlation.')
    xz = standardize(x[mask])
    yz = standardize(y[mask])
    return float(np.corrcoef(xz, yz)[0, 1])

print('Validation metrics available: RMSE, Spearman rho, and anomaly correlation (>=24 months).')
print('Important: independent product comparison is a plausibility check, not ground truth.')

# Sensitivity study plan
for name, values in [('orbit', ['ASCENDING', 'DESCENDING']), ('quantile', [0.05, 0.10, 0.15]), ('scale', [10, 50, 100])]:
    print(f'{name}: {values}')
print('Compare province mean series, valid-month coverage and edge fit across scenarios.')

## 7. Outputs and exports
Purpose: export the final relative index with clear metadata and a consistent projected CRS.

In [ ]:
# Cell 9: export placeholders and file setup
out_dir = Path(CFG['export_dir'])
out_dir.mkdir(exist_ok=True, parents=True)

csv_path = out_dir / 'province_mean_index.csv'
pd.DataFrame({
    'date': pd.Series([], dtype='datetime64[ns]'),
    'province_mean_index': pd.Series([], dtype=float),
    'dry_edge': pd.Series([], dtype=float),
    'wet_edge': pd.Series([], dtype=float),
}).to_csv(csv_path, index=False)
print(f'CSV export template written: {csv_path}')

# Optional GeoTIFF export:
# geemap.ee_export_image(monthly_mean, filename=str(out_dir / 'sar_relative_index.tif'), scale=CFG['scale'], region=study_area, crs=CFG['crs'])
print('GeoTIFF export is optional and should be run only after the final map is confirmed.')

# NetCDF generation is done after the final index array is assembled and clipped.
print('NetCDF export is ready once the final monthly or period mean array is assembled.')

## 8. Final assumptions and next research steps
Purpose: be explicit about what this method does and does not support.

In [ ]:
# Cell 10: final summary for the user
print('Assumptions: fixed regional dry/wet edges, linear-unit fitting, no zero-filling for missing data, and no conversion to volumetric soil moisture.')
print('Limitations: vegetation, roughness, topography, and water bodies can strongly influence the envelope.')
print('Recommended next steps: in-situ validation, independent product comparison by DATE, cross-orbit sensitivity analysis, and local calibration if converting to physical soil moisture units.')